# TP 02 Blockchain : Preuve d'enjeu (Proof of Stake)

Ce notebook principal realise la simulation complete du protocole Proof of Stake (PoS) :
- **Etapes 1 a 10** : Application rigoureuse des regles R1 a R7 (Mise, Selection, Bloc unique, Tirage public, Recompense, Validite, Sanction Slashing).
- **Section 4 : Tests du reseau** :
  1. Convergence statistique selon la taille de l'echantillon (500 vs 100 000 blocs).
  2. Impact de l'arrivee de nouveaux validateurs.
  3. Risque de centralisation lie au taux de recompense (taux = 1.0%).
  4. Detection d'un tricheur habile par controle frequentiel.


## Connexion au reseau
Un seul participant (ou le chef d'orchestre) lance le `/reset` pour reinitialiser la blockchain et fixer la graine aleatoire a 2 afin de garantir la reproductibilite exacte des resultats.


In [1]:
import requests
import random

RESEAU = "http://127.0.0.1:5000"

# Reinitialisation du serveur Flask
reponse = requests.post(RESEAU + "/reset").json()
print("POST /reset :", reponse)


POST /reset : {'message': 'reseau vide, graine 2'}


## Etape 1 : Rejoindre (R1)
Chaque participant entre dans le reseau en bloquant une mise >= 10 ETH (R1).
Les 4 participants initiaux sont :
- **Idriss** : 40 ETH
- **Moussa** : 25 ETH
- **Youssef** : 20 ETH
- **Ayoub** : 15 ETH

Une mise inferieure au seuil minimal de 10 ETH est rejetee avec le code HTTP 400.


In [2]:
participants = [
    ("Idriss", 40),
    ("Moussa", 25),
    ("Youssef", 20),
    ("Ayoub", 15)
]

for nom, mise in participants:
    rep = requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})
    print(rep.status_code, rep.json())

# Test de verification de la regle R1 avec une mise invalide (< 10 ETH)
rep_invalide = requests.post(RESEAU + "/rejoindre", json={"nom": "Attaquant", "mise": 5})
print("\nTest mise invalide (5 ETH) :", rep_invalide.status_code, rep_invalide.json())


200 {'mise_totale': 40, 'validateur': {'blocs': 0, 'exclu': False, 'mise': 40, 'nom': 'Idriss'}}
200 {'mise_totale': 65, 'validateur': {'blocs': 0, 'exclu': False, 'mise': 25, 'nom': 'Moussa'}}
200 {'mise_totale': 85, 'validateur': {'blocs': 0, 'exclu': False, 'mise': 20, 'nom': 'Youssef'}}
200 {'mise_totale': 100, 'validateur': {'blocs': 0, 'exclu': False, 'mise': 15, 'nom': 'Ayoub'}}

Test mise invalide (5 ETH) : 400 {'erreur': 'mise inferieure a 10'}


## Etape 2 : Les parts (R2)
Recuperation de l'etat des validateurs aupres du serveur (`GET /validateurs`).
Calcul de la mise totale $M = \sum m_j$ et de la part relative de chaque validateur $p_i = m_i / M$.


In [3]:
def parts(mises):
    """Part de chaque validateur dans la mise totale."""
    total = sum(m for _, m in mises)
    if total == 0:
        return {nom: 0.0 for nom, _ in mises}
    return {nom: 100.0 * m / total for nom, m in mises}

validateurs = requests.get(RESEAU + "/validateurs").json()
total = sum(v["mise"] for v in validateurs)
print(f"Mise totale : {total:.0f} ETH\n")

print(f"{'Validateur':<10} {'Mise':>8} {'Part':>12}")
for v in validateurs:
    part_pct = 100.0 * v["mise"] / total
    print(f"{v['nom']:<10} {v['mise']:>5.0f} ETH {part_pct:>10.1f} %")


Mise totale : 100 ETH

Validateur     Mise         Part
Idriss        40 ETH       40.0 %
Moussa        25 ETH       25.0 %
Youssef       20 ETH       20.0 %
Ayoub         15 ETH       15.0 %


## Etape 3 : Le tirage (R2)
Recopie locale de la regle de selection `choisir_proposant` :
Sur un segment de longueur $M$, chaque validateur occupe un intervalle correspondant a sa mise.
Le tirage $t \in [0; M]$ selectionne le premier validateur dont le cumul des mises depasse $t$.
Test avec les tirages du sujet : 52.3, 12.0 et 95.5.


In [4]:
def choisir_proposant(mises, tirage):
    """R2 : le validateur dont l'intervalle de mise contient le tirage."""
    cumul = 0.0
    for nom, mise in mises:
        cumul += mise
        if tirage <= cumul:
            return nom
    return mises[-1][0] if mises else None

mises = [("Idriss", 40), ("Moussa", 25), ("Youssef", 20), ("Ayoub", 15)]
for t in [52.3, 12.0, 95.5]:
    print(f"tirage = {t:<4} -> proposant : {choisir_proposant(mises, t)}")


tirage = 52.3 -> proposant : Moussa
tirage = 12.0 -> proposant : Idriss
tirage = 95.5 -> proposant : Ayoub


## Etape 4 : Un bloc (R3, R4)
Production du premier bloc par le serveur via `POST /produire_bloc`.
Le bloc transporte publiquement son tirage et les mises du moment (R4).
Chaque participant recalcule le proposant attendu pour valider le respect de R2.


In [5]:
bloc = requests.post(RESEAU + "/produire_bloc", json={}).json()
print("Bloc produit :", bloc)

recalcul = choisir_proposant(bloc["mises"], bloc["tirage"])
print("Recalcul chez moi :", recalcul)
print("Concordance :", recalcul == bloc["proposant"])


Bloc produit : {'mises': [['Idriss', 40], ['Moussa', 25], ['Youssef', 20], ['Ayoub', 15]], 'numero': 1, 'proposant': 'Ayoub', 'tirage': 95.6}
Recalcul chez moi : Ayoub
Concordance : True


## Etape 5 : 14 blocs de plus (R3)
Production de 14 blocs supplementaires pour observer les 15 premiers blocs de la blockchain.


In [6]:
requests.post(RESEAU + "/produire_blocs", json={"n": 14})
chaine = requests.get(RESEAU + "/chaine").json()

print(f"{'Bloc':>4} {'Proposant':<10} {'Tirage':>7}")
for b in chaine:
    print(f"{b['numero']:>4} {b['proposant']:<10} {b['tirage']:>7.2f}")


Bloc Proposant   Tirage
   1 Ayoub        95.60
   2 Ayoub        94.78
   3 Idriss        5.66
   4 Idriss        8.49
   5 Youssef      83.55
   6 Youssef      73.60
   7 Youssef      66.97
   8 Idriss       30.81
   9 Moussa       60.59
  10 Moussa       60.68
  11 Moussa       58.12
  12 Idriss       15.84
  13 Moussa       43.07
  14 Idriss       39.35
  15 Youssef      72.30


## Etape 6 : Proportionnalite sur 5 000 blocs (R2)
Production de 5 000 blocs pour verifier empiriquement que la frequence de selection converge vers la part de mise ($p_i = m_i / M$).


In [7]:
def afficher_parts(validateurs, chaine):
    """Tableau part de mise / part des blocs, avec barres de '#' ASCII."""
    total = sum(v["mise"] for v in validateurs)
    print(f"taille de la chaine : {len(chaine)}")
    print(f"{'Validateur':<10} {'Part de mise':>14} {'Part des blocs':>16}")
    for v in validateurs:
        p_m = 100.0 * v["mise"] / total
        p_b = 100.0 * v["blocs"] / len(chaine)
        barre = "#" * int(p_b / 2)
        print(f"{v['nom']:<10} {p_m:>12.1f} % {p_b:>14.1f} %  {barre}")

requests.post(RESEAU + "/produire_blocs", json={"n": 5000})
validateurs = requests.get(RESEAU + "/validateurs").json()
chaine = requests.get(RESEAU + "/chaine").json()

afficher_parts(validateurs, chaine)


taille de la chaine : 5015
Validateur   Part de mise   Part des blocs
Idriss             40.0 %           39.5 %  ###################
Moussa             25.0 %           25.2 %  ############
Youssef            20.0 %           20.7 %  ##########
Ayoub              15.0 %           14.6 %  #######


## Etape 7 : La recompense (R5)
Etude de l'effet d'une recompense de bloc de 0.3% ($r = 0.003$) sur 1 000 blocs.
Analyse de l'amplification de mise et de la dynamique de centralisation inherente au PoS simple.


In [8]:
requests.post(RESEAU + "/reset")
for nom, mise in participants:
    requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})

depart = {v["nom"]: v["mise"] for v in requests.get(RESEAU + "/validateurs").json()}
requests.post(RESEAU + "/produire_blocs", json={"n": 1000, "taux": 0.003})
validateurs = requests.get(RESEAU + "/validateurs").json()

total_d = sum(depart.values())
total_f = sum(v["mise"] for v in validateurs)

print(f"{'Validateur':<10} {'Mise depart':>12} {'Mise fin':>10} {'Part depart':>13} {'Part fin':>11} {'Blocs':>7}")
for v in validateurs:
    nom = v["nom"]
    p_dep = 100.0 * depart[nom] / total_d
    p_fin = 100.0 * v["mise"] / total_f
    print(f"{nom:<10} {depart[nom]:>12.1f} {v['mise']:>10.1f} {p_dep:>11.1f} % {p_fin:>9.1f} % {v['blocs']:>7}")
print(f"{'Reseau':<10} {total_d:>12.1f} {total_f:>10.1f}\n")

print(f"{'Validateur':<10} {'Mise x':>8} {'Part depart -> Part fin':>26}")
for v in validateurs:
    nom = v["nom"]
    facteur = v["mise"] / depart[nom]
    p_dep = 100.0 * depart[nom] / total_d
    p_fin = 100.0 * v["mise"] / total_f
    print(f"{nom:<10} {facteur:>8.2f} {p_dep:>11.1f} % -> {p_fin:>5.1f} %")
print(f"{'Reseau':<10} {total_f / total_d:>8.2f}")


Validateur  Mise depart   Mise fin   Part depart    Part fin   Blocs
Idriss             40.0      175.2        40.0 %      62.9 %     493
Moussa             25.0       50.2        25.0 %      18.0 %     233
Youssef            20.0       31.3        20.0 %      11.2 %     149
Ayoub              15.0       21.8        15.0 %       7.8 %     125
Reseau            100.0      278.5

Validateur   Mise x    Part depart -> Part fin
Idriss         4.38        40.0 % ->  62.9 %
Moussa         2.01        25.0 % ->  18.0 %
Youssef        1.56        20.0 % ->  11.2 %
Ayoub          1.45        15.0 % ->   7.8 %
Reseau         2.78


## Etape 8 : Falsifier (R6)
Simulation d'une falsification : Ayoub injecte un bloc signe a son propre nom alors que le tirage aleatoire designe un autre validateur.


In [9]:
requests.post(RESEAU + "/reset")
for nom, mise in participants:
    requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})

requests.post(RESEAU + "/produire_blocs", json={"n": 5})
faux = requests.post(RESEAU + "/falsifier", json={"nom": "Ayoub"}).json()
requests.post(RESEAU + "/produire_blocs", json={"n": 3})

recalcul_faux = choisir_proposant(faux["mises"], faux["tirage"])
print(f"bloc {faux['numero']} signe par {faux['proposant']} | tirage {faux['tirage']}")
print(f"recalcul : {recalcul_faux}")


bloc 6 signe par Ayoub | tirage 73.6
recalcul : Youssef


## Etape 9 : Detecter (R6)
Chaque nœud audite la blockchain de maniere autonome en appliquant `verifier_bloc` sur chaque bloc telecharge.


In [10]:
def verifier_bloc(bloc):
    """R6 : le nom inscrit doit correspondre au proposant calcule par R2."""
    return bloc["proposant"] == choisir_proposant(bloc["mises"], bloc["tirage"])

def verifier_chaine(chaine):
    """R6 : applique verifier_bloc a toute la chaine."""
    return [b["numero"] for b in chaine if not verifier_bloc(b)]

chaine = requests.get(RESEAU + "/chaine").json()

print(f"{'Bloc':>4} {'Signe par':<10} {'Tirage':>7} {'Recalcul':<10} {'Verdict'}")
for b in chaine:
    recalc = choisir_proposant(b["mises"], b["tirage"])
    verdict = "ok" if verifier_bloc(b) else "FALSIFIE"
    print(f"{b['numero']:>4} {b['proposant']:<10} {b['tirage']:>7.2f} {recalc:<10} {verdict}")

rep_serveur = requests.get(RESEAU + "/verifier").json()
print("\nreponse du reseau :", rep_serveur["blocs_falsifies"])
print("recalcul local    :", verifier_chaine(chaine))


Bloc Signe par   Tirage Recalcul   Verdict
   1 Ayoub        95.60 Ayoub      ok
   2 Ayoub        94.78 Ayoub      ok
   3 Idriss        5.66 Idriss     ok
   4 Idriss        8.49 Idriss     ok
   5 Youssef      83.55 Youssef    ok
   6 Ayoub        73.60 Youssef    FALSIFIE
   7 Youssef      66.97 Youssef    ok
   8 Idriss       30.81 Idriss     ok
   9 Moussa       60.59 Moussa     ok

reponse du reseau : [6]
recalcul local    : [6]


## Etape 10 : Sanctionner (R7 - Slashing)
Le protocole applique le slashing : confiscation de 50% de la mise du tricheur et exclusion definitive de la liste active des validateurs.


In [11]:
for numero in rep_serveur["blocs_falsifies"]:
    coupable = chaine[numero - 1]["proposant"]
    r = requests.post(RESEAU + "/sanctionner", json={"nom": coupable, "fraction": 0.5}).json()
    print(f"{coupable} : {r}")

validateurs_actifs = [v for v in requests.get(RESEAU + "/validateurs").json() if not v["exclu"]]
total_restant = sum(v["mise"] for v in validateurs_actifs)
print(f"mise totale apres exclusion : {total_restant:.1f} ETH")


Ayoub : {'exclu': True, 'mise_restante': 7.5, 'perte': 7.5}
mise totale apres exclusion : 85.0 ETH


## Section 4 : Tests approfondis du reseau

### 1. La taille de l'echantillon (500 vs 100 000 blocs)
Illustration de la loi des grands nombres : l'ecart maximal diminue de facon significative quand l'echantillon croit.


In [12]:
for n_blocs in [500, 100000]:
    requests.post(RESEAU + "/reset")
    for nom, mise in participants:
        requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})
    requests.post(RESEAU + "/produire_blocs", json={"n": n_blocs})
    v_list = requests.get(RESEAU + "/validateurs").json()
    tot = sum(v["mise"] for v in v_list)
    print(f"\nAvec {n_blocs} blocs :")
    print(f"{'Validateur':<10} {'Part de mise':>14} {'Part des blocs':>16}")
    for v in v_list:
        p_m = 100.0 * v["mise"] / tot
        p_b = 100.0 * v["blocs"] / n_blocs
        barre = "#" * int(p_b / 2)
        print(f"{v['nom']:<10} {p_m:>12.1f} % {p_b:>14.1f} %  {barre}")


Avec 500 blocs :
Validateur   Part de mise   Part des blocs
Idriss             40.0 %           39.4 %  ###################
Moussa             25.0 %           24.6 %  ############
Youssef            20.0 %           20.0 %  ##########
Ayoub              15.0 %           16.0 %  ########

Avec 100000 blocs :
Validateur   Part de mise   Part des blocs
Idriss             40.0 %           39.9 %  ###################
Moussa             25.0 %           25.0 %  ############
Youssef            20.0 %           20.1 %  ##########
Ayoub              15.0 %           15.0 %  #######


### 2. L'arrivee de 3 validateurs
Entree d'Aymen (100 ETH), Timan (50 ETH) et Meryem (50 ETH) dans le reseau.


In [13]:
requests.post(RESEAU + "/reset")
participants_7 = participants + [
    ("Aymen", 100),
    ("Timan", 50),
    ("Meryem", 50)
]
for nom, mise in participants_7:
    requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})

requests.post(RESEAU + "/produire_blocs", json={"n": 5000})
v_list = requests.get(RESEAU + "/validateurs").json()
tot = sum(v["mise"] for v in v_list)

print(f"{'Validateur':<10} {'Part de mise':>14} {'Part des blocs':>16}")
for v in v_list:
    p_m = 100.0 * v["mise"] / tot
    p_b = 100.0 * v["blocs"] / 5000
    barre = "#" * int(p_b / 2)
    print(f"{v['nom']:<10} {p_m:>12.1f} % {p_b:>14.1f} %  {barre}")


Validateur   Part de mise   Part des blocs
Idriss             13.3 %           13.2 %  ######
Moussa              8.3 %            8.2 %  ####
Youssef             6.7 %            6.8 %  ###
Ayoub               5.0 %            4.9 %  ##
Aymen              33.3 %           33.4 %  ################
Timan              16.7 %           17.3 %  ########
Meryem             16.7 %           16.3 %  ########


### 3. Une recompense plus forte
Observation de l'effet d'avalanche de centralisation avec un taux de 1.0% sur 1 000 blocs.


In [14]:
requests.post(RESEAU + "/reset")
for nom, mise in participants:
    requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})

dep3 = {v["nom"]: v["mise"] for v in requests.get(RESEAU + "/validateurs").json()}
requests.post(RESEAU + "/produire_blocs", json={"n": 1000, "taux": 0.01})
v3 = requests.get(RESEAU + "/validateurs").json()
tot_d = sum(dep3.values())
tot_f = sum(v["mise"] for v in v3)

print(f"{'Validateur':<10} {'Mise depart':>12} {'Mise fin':>10} {'Part depart':>13} {'Part fin':>11} {'Blocs':>7}")
for v in v3:
    nom = v["nom"]
    p_dep = 100.0 * dep3[nom] / tot_d
    p_fin = 100.0 * v["mise"] / tot_f
    print(f"{nom:<10} {dep3[nom]:>12.1f} {v['mise']:>10.1f} {p_dep:>11.1f} % {p_fin:>9.1f} % {v['blocs']:>7}")
print(f"{'Reseau':<10} {tot_d:>12.1f} {tot_f:>10.1f}")


Validateur  Mise depart   Mise fin   Part depart    Part fin   Blocs
Idriss             40.0    93914.3        40.0 %      99.9 %     780
Moussa             25.0       69.0        25.0 %       0.1 %     102
Youssef            20.0       37.1        20.0 %       0.0 %      62
Ayoub              15.0       26.2        15.0 %       0.0 %      56
Reseau            100.0    94046.5


### 4. Un tricheur plus habile
Ayoub forge un tirage truque dans son propre intervalle [85; 100].
Le test bloc par bloc est impuissant, mais l'analyse des frequences revele l'anomalie statistique.


In [15]:
requests.post(RESEAU + "/reset")
for nom, mise in participants:
    requests.post(RESEAU + "/rejoindre", json={"nom": nom, "mise": mise})

random.seed(2)
debut = 85
fin = 100
for i in range(400):
    if i % 4 == 0:
        tirage = round(random.uniform(debut, fin), 2)
        requests.post(RESEAU + "/falsifier", json={"nom": "Ayoub", "tirage": tirage})
    else:
        requests.post(RESEAU + "/produire_bloc", json={})

rep = requests.get(RESEAU + "/verifier").json()
print("blocs falsifies :", rep["blocs_falsifies"])
print(f"{'Validateur':<10} {'Part de mise':>14} {'Part des blocs':>16} {'Verdict':>10}")
for f in rep["frequences"]:
    verdict = "SUSPECT" if f["suspect"] else "normal"
    print(f"{f['nom']:<10} {f['part_mise']:>12.1f} % {f['part_blocs']:>14.1f} % {verdict:>10}")


blocs falsifies : []
Validateur   Part de mise   Part des blocs    Verdict
Idriss             40.0 %           28.8 %     normal
Moussa             25.0 %           18.8 %     normal
Youssef            20.0 %           16.2 %     normal
Ayoub              15.0 %           36.2 %    SUSPECT
